In [5]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping


seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

BASE_DIR = Path().resolve().parent

data_path = BASE_DIR / "data_processed/NASA/nasa_battery_features_rolling.csv"
results_root = BASE_DIR / "results/AE_NASA"
results_root.mkdir(parents=True, exist_ok=True)

encoding_dim_list = [4, 8]
hidden_dim_list = [16, 32]
batch_size_list = [32, 64]
learning_rate_list = [1e-3, 5e-4]
threshold_quantile_list = [0.25, 0.30, 0.35, 0.40, 0.50]

epochs = 100
patience = 10
window_size = 50

train_limit = 5000
test_normal_limit = 5000
test_anomaly_limit = 2000

df = pd.read_csv(data_path)

feature_cols = [
    "V", "I", "T",
    "Current_load", "Voltage_load", "Time",
    "V_diff1", "I_diff1", "T_diff1",
    "V_roll_mean", "V_roll_std", "V_roll_min", "V_roll_max",
    "I_roll_mean", "I_roll_std",
    "T_roll_mean", "T_roll_std"
]

required_cols = feature_cols + ["y_true_file"]
df = df.dropna(subset=required_cols).reset_index(drop=True)

X = df[feature_cols].copy()
y_true = df["y_true_file"].astype(int).values


def get_window_labels(y_array, window_size):
    y_windows = []
    for i in range(len(y_array) - window_size + 1):
        y_windows.append(1 if np.any(y_array[i:i + window_size] == 1) else 0)
    return np.array(y_windows)


def build_windows_by_indices(X_array, indices, window_size):
    windows = []
    for i in indices:
        windows.append(X_array[i:i + window_size].reshape(-1))
    return np.array(windows, dtype=np.float32)


def to_str(value):
    if isinstance(value, str):
        return value
    return str(value).replace(".", "_")


def build_autoencoder(input_dim, hidden_dim, encoding_dim, learning_rate):
    inputs = Input(shape=(input_dim,))
    x = Dense(hidden_dim, activation="relu")(inputs)
    latent = Dense(encoding_dim, activation="relu")(x)
    x = Dense(hidden_dim, activation="relu")(latent)
    outputs = Dense(input_dim, activation="linear")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return model


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

y_windows = get_window_labels(y_true, window_size)

normal_window_indices = np.where(y_windows == 0)[0]
anomaly_window_indices = np.where(y_windows == 1)[0]

rng = np.random.default_rng(seed)

if len(normal_window_indices) > train_limit:
    train_indices = rng.choice(normal_window_indices, size=train_limit, replace=False)
else:
    train_indices = normal_window_indices.copy()

if len(normal_window_indices) > test_normal_limit:
    test_normal_indices = rng.choice(normal_window_indices, size=test_normal_limit, replace=False)
else:
    test_normal_indices = normal_window_indices.copy()

if len(anomaly_window_indices) > test_anomaly_limit:
    test_anomaly_indices = rng.choice(anomaly_window_indices, size=test_anomaly_limit, replace=False)
else:
    test_anomaly_indices = anomaly_window_indices.copy()

X_train = build_windows_by_indices(X_scaled, train_indices, window_size)

test_indices = np.concatenate([test_normal_indices, test_anomaly_indices])
y_test = np.array(
    [0] * len(test_normal_indices) + [1] * len(test_anomaly_indices)
)

shuffle_indices = rng.permutation(len(test_indices))
test_indices = test_indices[shuffle_indices]
y_test = y_test[shuffle_indices]

X_test = build_windows_by_indices(X_scaled, test_indices, window_size)

input_dim = X_train.shape[1]

all_results = []

for hidden_dim in hidden_dim_list:
    hidden_dir = results_root / f"hidden_{to_str(hidden_dim)}"
    hidden_dir.mkdir(parents=True, exist_ok=True)

    hidden_results = []

    for encoding_dim in encoding_dim_list:
        encoding_dir = hidden_dir / f"latent_{to_str(encoding_dim)}"
        encoding_dir.mkdir(parents=True, exist_ok=True)

        encoding_results = []

        for batch_size in batch_size_list:
            batch_dir = encoding_dir / f"batch_{to_str(batch_size)}"
            batch_dir.mkdir(parents=True, exist_ok=True)

            batch_results = []

            for learning_rate in learning_rate_list:
                lr_dir = batch_dir / f"lr_{to_str(learning_rate)}"
                lr_dir.mkdir(parents=True, exist_ok=True)

                tf.keras.backend.clear_session()

                model = build_autoencoder(
                    input_dim=input_dim,
                    hidden_dim=hidden_dim,
                    encoding_dim=encoding_dim,
                    learning_rate=learning_rate
                )

                early_stopping = EarlyStopping(
                    monitor="val_loss",
                    patience=patience,
                    restore_best_weights=True
                )

                history = model.fit(
                    X_train,
                    X_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_split=0.2,
                    shuffle=True,
                    callbacks=[early_stopping],
                    verbose=0
                )

                train_recon = model.predict(X_train, verbose=0)
                train_errors = np.mean(np.square(X_train - train_recon), axis=1)

                test_recon = model.predict(X_test, verbose=0)
                test_errors = np.mean(np.square(X_test - test_recon), axis=1)

                for threshold_quantile in threshold_quantile_list:
                    exp_name = (
                        f"window_{window_size}"
                        f"_latent_{to_str(encoding_dim)}"
                        f"_hidden_{to_str(hidden_dim)}"
                        f"_batch_{to_str(batch_size)}"
                        f"_lr_{to_str(learning_rate)}"
                        f"_q_{to_str(threshold_quantile)}"
                    )

                    exp_dir = lr_dir / f"q_{to_str(threshold_quantile)}"
                    exp_dir.mkdir(parents=True, exist_ok=True)

                    threshold = float(np.quantile(train_errors, threshold_quantile))
                    y_pred = (test_errors > threshold).astype(int)

                    cm = confusion_matrix(y_test, y_pred)

                    accuracy = accuracy_score(y_test, y_pred) * 100
                    precision = precision_score(y_test, y_pred, zero_division=0) * 100
                    recall = recall_score(y_test, y_pred, zero_division=0) * 100
                    f1 = f1_score(y_test, y_pred, zero_division=0) * 100

                    epochs_trained = len(history.history["loss"])
                    best_val_loss = float(np.min(history.history["val_loss"]))
                    final_train_loss = float(history.history["loss"][-1])

                    result_row = {
                        "experiment": exp_name,
                        "window_size": window_size,
                        "hidden_dim": hidden_dim,
                        "encoding_dim": encoding_dim,
                        "batch_size": batch_size,
                        "learning_rate": learning_rate,
                        "threshold_quantile": threshold_quantile,
                        "threshold": threshold,
                        "epochs_trained": epochs_trained,
                        "best_val_loss": round(best_val_loss, 8),
                        "final_train_loss": round(final_train_loss, 8),
                        "accuracy_%": round(accuracy, 2),
                        "precision_%": round(precision, 2),
                        "recall_%": round(recall, 2),
                        "f1_%": round(f1, 2)
                    }

                    all_results.append(result_row)
                    hidden_results.append(result_row)
                    encoding_results.append(result_row)
                    batch_results.append(result_row)

                    predictions_df = pd.DataFrame({
                        "window_start_index": test_indices,
                        "window_end_index": test_indices + window_size - 1,
                        "y_true": y_test,
                        "reconstruction_error": test_errors,
                        "threshold": threshold,
                        "is_anomaly": y_pred
                    })

                    predictions_df.to_csv(exp_dir / "predictions.csv", index=False)

                    history_df = pd.DataFrame(history.history)
                    history_df.to_csv(exp_dir / "history.csv", index=False)

                    metrics_df = pd.DataFrame({
                        "Metrika": [
                            "Presnosť",
                            "Precíznosť",
                            "Citlivosť",
                            "F1-skóre"
                        ],
                        "Hodnota (%)": [
                            round(accuracy, 2),
                            round(precision, 2),
                            round(recall, 2),
                            round(f1, 2)
                        ]
                    })

                    fig, ax = plt.subplots(figsize=(6, 2))
                    ax.axis("off")

                    table = ax.table(
                        cellText=metrics_df.values,
                        colLabels=metrics_df.columns,
                        loc="center",
                        cellLoc="center"
                    )

                    table.auto_set_font_size(False)
                    table.set_fontsize(12)
                    table.scale(1, 2)

                    for (row, col), cell in table.get_celld().items():
                        if row == 0:
                            cell.set_text_props(weight="bold")
                            cell.set_facecolor("#dce6f1")
                        else:
                            cell.set_facecolor("#f9f9f9")

                    plt.title("Výsledné metriky modelu autoenkódera – NASA", fontsize=12, pad=20)
                    plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
                    plt.close()

                    plt.figure(figsize=(6, 5))
                    plt.imshow(cm, interpolation="nearest")
                    plt.title("Matica zámen")
                    plt.colorbar()

                    classes = ["Normálne", "Anomálne"]
                    tick_marks = np.arange(len(classes))
                    plt.xticks(tick_marks, classes)
                    plt.yticks(tick_marks, classes)

                    threshold_cm = cm.max() / 2.0 if cm.max() > 0 else 0.5

                    for i in range(cm.shape[0]):
                        for j in range(cm.shape[1]):
                            color = "black" if cm[i, j] > threshold_cm else "white"
                            plt.text(
                                j, i, cm[i, j],
                                ha="center",
                                va="center",
                                color=color
                            )

                    plt.xlabel("Predikované triedy")
                    plt.ylabel("Skutočné triedy")
                    plt.tight_layout()
                    plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
                    plt.close()

                    epochs_axis = np.arange(1, len(history.history["loss"]) + 1)

                    plt.figure(figsize=(8, 4))
                    plt.plot(epochs_axis, history.history["loss"], label="Tréningová chyba")
                    plt.plot(epochs_axis, history.history["val_loss"], label="Validačná chyba")
                    plt.xlabel("Epocha")
                    plt.ylabel("Chyba")
                    plt.title("Priebeh učenia autoenkódera – NASA")
                    plt.legend()
                    plt.tight_layout()
                    plt.savefig(exp_dir / "loss_curve.png", dpi=300, bbox_inches="tight")
                    plt.close()

                    plt.figure(figsize=(10, 4))
                    plt.plot(np.arange(len(test_errors)), test_errors, linewidth=1)
                    plt.axhline(y=threshold, linestyle="--")
                    plt.xlabel("Index okna")
                    plt.ylabel("Chyba rekonštrukcie")
                    plt.title("Chyba rekonštrukcie – NASA")
                    plt.tight_layout()
                    plt.savefig(exp_dir / "reconstruction_error.png", dpi=300, bbox_inches="tight")
                    plt.close()

                batch_results_df = pd.DataFrame(batch_results)
                batch_results_df = batch_results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
                batch_results_df.to_csv(lr_dir / "summary_results.csv", index=False)

        encoding_results_df = pd.DataFrame(encoding_results)
        encoding_results_df = encoding_results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
        encoding_results_df.to_csv(encoding_dir / "summary_results.csv", index=False)

    hidden_results_df = pd.DataFrame(hidden_results)
    hidden_results_df = hidden_results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
    hidden_results_df.to_csv(hidden_dir / "summary_results.csv", index=False)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Done
